In [1]:
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = '/content/drive/MyDrive/урок 7/Лямбда-функции. Список транзакций.xlsx'
transaction_list = pd.read_excel(path)

Вы работаете аналитиком в сети супермакетов, и ваша задача - автоматизировать работу с торговыми точками.\
Ваша задача - исследовать различные торговые точки на предмет их отклонения от среднего чека.\
Если средний чек в торговой точке меньше, чем средний чек по всем наблюдениям, умноженный на определенное пороговое значение, то торговая точка помечается (как Булево значение False).\
Также необходимо работать только с теми торговыми точками, в которых достаточное количество наблюдений, чтобы "верить" среднему чеку.

### Задание 1

Импортируйте файл **'Лямбда-функции. Список транзакций.xlsx'** \
Рассчитайте среднее значение чека по всем операциям в таблице. \
Рассчитайте среднее значение чека и количество транзакций для каждой торговой точки по отдельности.


In [5]:
transaction_list.head()

,id_client,id_point,check
0,1034834,106,6641
1,1034835,106,3134
2,1034836,103,6449
3,1034837,109,3293
4,1034838,109,1038


In [6]:
avg_check = transaction_list['check'].mean()
print(f"Среднее значение чека по всем операциям в таблице {avg_check:.2f}")

Среднее значение чека по всем операциям в таблице 3930.85


In [7]:
total = transaction_list.groupby('id_point')['check'].agg(['mean','count'])
total = total.rename(columns={'mean':'avg_check','count':'transaction_count'})
total

,avg_check,transaction_count
id_point,,
101,4038.350852,704
102,5335.203008,532
103,3857.566596,473
104,3893.656000,375
105,3883.217064,797
106,5065.804348,736
107,2936.442130,432
108,3025.939058,361
109,2098.698876,445


### Задание 2

Создайте функцию, которая на вход будет брать следующие аргументы:\
\
    1. Средний чек, который мы проверяем \
    2. Кол-во транзакций, которое мы проверяем \
    3. Пороговое значение среднего чека \
    4. Пороговое значение кол-ва транзакций \
    5. Процент отступления от порогового значения среднего чека (со значением по умолчанию 0.9)

Если количество наблюдений меньше чем пороговое значение количества наблюдений, то функция принтит "Мало наблюдений!".

Если проверяемый средний чек меньше, чем пороговый средний чек, умноженный на "процент отступления", то функция возвращает False.

В противном случае функция возвращает True.

Также с помощью выражения except обработайте ошибку типизации данных (TypeError), выведите сообщение "Нужны цифры!".

Протестируйте функцию на параметрах торговой точки 101 (4038, 704).\
В качестве порогового значения количества наблюдений возьмите 300.\
В качестве порогового значения среднего чека возьмите рассчитанный в прошлом пункте средний чек по всем торговым точкам вместе.

In [19]:
def check_store(avg_check, cnt_transactions, tresh_avg_check, thresh_cnt_transactions, dev_percent=0.9):
  try:
  #Проверка на количество наблюдений
    if cnt_transactions < thresh_cnt_transactions:
      print("Мало наблюдений!")
  #Проверка порогового значения среднего чека
    if avg_check < (tresh_avg_check * dev_percent):
      return False

    return True
  except TypeError:
    print("Нужны цифры!")
    return None



In [20]:
result = check_store(avg_check=4038,cnt_transactions=704,tresh_avg_check=3930,thresh_cnt_transactions=300)

print(f"Результат функции для ТТ 101: {result}")

Результат функции для ТТ 101: True


In [17]:
cnt_thr = 300

### Задание 3

С помощью операции маппинга (mapping) примените данную функцию ко всем торговым точкам из датасета.\
Для этого создайте в сгруппированном датафрейме две дополнительные колонки: одну заполните пороговым значением среднего чека, а другую - пороговым значением количества наблюдений.\
После этого можно передать 4 колонки сгруппированного датафрейма в качестве аргументов функции map, наравне с вашей функций.\
Сохраните результат в список.\
Рассчитайте долю прошедших проверку (True) торговых точек. Для этого разделите сумму значений списка на его длину.

In [21]:
#Создаем новые колонки в таблице
total['tresh_avg_check'] = avg_check
total['thresh_cnt_transactions'] = cnt_thr
total

,avg_check,transaction_count,tresh_avg_check,thresh_cnt_transactions
id_point,,,,
101,4038.350852,704,3930.846756,300
102,5335.203008,532,3930.846756,300
103,3857.566596,473,3930.846756,300
104,3893.656000,375,3930.846756,300
105,3883.217064,797,3930.846756,300
106,5065.804348,736,3930.846756,300
107,2936.442130,432,3930.846756,300
108,3025.939058,361,3930.846756,300
109,2098.698876,445,3930.846756,300


In [28]:
# Применяем функцию с помощью встроенного map() к 4 колонкам
results_list = list(map(
    check_store,
    total['avg_check'],               # Колонка с текущим средним чеком
    total['transaction_count'],       # Колонка с текущим количеством транзакций
    total['tresh_avg_check'],         # Новая колонка с порогом чека
    total['thresh_cnt_transactions']  # Новая колонка с порогом транзакций
))
print(results_list)
# Расчет доли торговых точек, прошедших проверку (True)
results_list = sum(results_list) / len(results_list)

print(f"Доля торговых точек, прошедших проверку: {results_list:.2%}")

[True, True, True, True, True, True, False, False, False]
Доля торговых точек, прошедших проверку: 66.67%
